# Ingestão Bronze: ADLS Gen2 (raw) ➔ Delta Lake (squad1/bronze)
Executa uma rodada pontual de ingestão:
- Varre arquivos pendentes em `raw/real-time-data`[cite: 2]
- Grava os dados em modo `append` particionados por ano, mês, dia e hora em `squad1/bronze/ecommerce_enderecos`[cite: 2]
- Retorna o total de arquivos ingeridos para o orquestrador[cite: 2]

In [0]:
import os
from dotenv import load_dotenv
from pyspark.errors import AnalysisException
from pyspark.sql import functions as F, types as T
from pathlib import Path

# 1. Carregamento de Parâmetros e Credenciais
load_dotenv(".env")

client_id            = os.getenv("ADLS_CLIENT_ID")
tenant_id            = os.getenv("ADLS_TENANT_ID")
client_secret        = os.getenv("ADLS_CLIENT_SECRET")
# Usa a variável de ambiente se existir, senão assume o nome padrão do ambiente
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
source_container     = os.getenv("ADLS_CONTAINER_NAME", "raw")

# Diagnóstico detalhado caso falte alguma credencial
variaveis = {
    "ADLS_CLIENT_ID": client_id,
    "ADLS_TENANT_ID": tenant_id,
    "ADLS_CLIENT_SECRET": client_secret,
    "ADLS_STORAGE_ACCOUNT_NAME": storage_account_name
}
faltantes = [nome for nome, valor in variaveis.items() if not valor]

assert not faltantes, f"Faltou variável do ADLS no .env: {', '.join(faltantes)}"

# 2. Configurações para Serverless Compute via options
adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

source_glob = f"abfss://{source_container}@{storage_account_name}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_enderecos.parquet"
squad_base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
bronze_path = f"{squad_base}/bronze/ecommerce_enderecos"

def ler_delta(path):
    try:
        df = spark.read.format("delta").options(**adls_opts).load(path)
        df.schema
        return df
    except AnalysisException as e:
        if any(m in str(e) for m in ("PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST", "DELTA_MISSING_DELTA_TABLE", "is not a Delta table")):
            return None
        raise

target_schema = T.StructType.fromDDL(
    "id_endereco bigint, id_cliente bigint, logradouro string, numero bigint, "
    "complemento string, bairro string, cep string, cidade string, estado string, "
    "latitude double, longitude double, apelido string, is_principal boolean"
)

In [0]:
# 3. Checkpoint e Funções Auxiliares de Leitura
df_bronze_atual = ler_delta(bronze_path)

if df_bronze_atual is None:
    control_checkpoint = set()
    print("ℹ Bronze ainda não existe no storage. Será criada na primeira gravação.")
else:
    control_checkpoint = {
        r["bronze_source_file"]
        for r in df_bronze_atual.select("bronze_source_file").distinct().collect()
    }
    print(f"✔ Checkpoint: {len(control_checkpoint)} arquivo(s) já ingerido(s).")

def listar_arquivos_origem():
    try:
        arquivos = spark.read.format("binaryFile").options(**adls_opts).load(source_glob).select("path").collect()
    except AnalysisException as e:
        if "PATH_NOT_FOUND" in str(e):
            return []
        raise
    return sorted(r.path for r in arquivos)

def ler_micro_lote(file_path):
    df = spark.read.options(**adls_opts).parquet(file_path)

    esperadas = [c.name for c in target_schema.fields]
    faltando = set(esperadas) - set(df.columns)
    extras = set(df.columns) - set(esperadas)
    if faltando or extras:
        raise ValueError(f"Schema divergente. Faltando: {sorted(faltando)} | Extras: {sorted(extras)}")

    for campo in target_schema.fields:
        tipo_origem = df.schema[campo.name].dataType
        if tipo_origem != campo.dataType and df.filter(F.col(campo.name).isNotNull()).limit(1).count() > 0:
            raise ValueError(f"Coluna '{campo.name}' veio com tipo divergente: {tipo_origem}.")

    return df.select([F.col(c.name).cast(c.dataType) for c in target_schema.fields])

In [0]:
# ==============================================================================
# 4. Execução Pontual da Bronze com retorno detalhado em JSON
# ==============================================================================
import json

new_files = [p for p in listar_arquivos_origem() if p not in control_checkpoint]
print(f"Arquivos novos encontrados para Bronze: {len(new_files)}")

sucessos_arquivos = []

for file_path in new_files:
    try:
        print(f"Ingerindo Bronze: {file_path}")
        df_bronze = (
            ler_micro_lote(file_path)
            .withColumn("bronze_ingested_at", F.current_timestamp())
            .withColumn("bronze_source_file", F.lit(file_path))
            .withColumn("ano",  F.year("bronze_ingested_at"))
            .withColumn("mes",  F.month("bronze_ingested_at"))
            .withColumn("dia",  F.dayofmonth("bronze_ingested_at"))
            .withColumn("hora", F.hour("bronze_ingested_at"))
        )

        (
            df_bronze.write
            .format("delta")
            .mode("append")
            .options(**adls_opts)
            .partitionBy("ano", "mes", "dia", "hora")
            .save(bronze_path)
        )

        # Guarda o caminho do arquivo processado com sucesso
        sucessos_arquivos.append(file_path)
        print(f"✔ Sucesso na Bronze.")
    except Exception as e:
        print(f"✖ Falha em '{file_path}': {e}")

# Prepara o resumo estruturado para o orquestrador
resultado_bronze = {
    "qtd_ingeridos": len(sucessos_arquivos),
    "arquivos": sucessos_arquivos
}

# Retorna a string JSON para o dbutils.notebook.run() do orquestrador
dbutils.notebook.exit(json.dumps(resultado_bronze))